# Step 3 -- Ancestral GOEA with EAGLE (Extant and Ancestral Gene List Enrichment)

## What is EAGLE?

EAGLE (Extant and Ancestral Gene List Enrichment) performs Gene Ontology Enrichment Analysis (GOEA) on evolutionary events inferred from HOGs.

EAGLE can tell us which functions are enriched in a list of genes, and even more specifically:

> Which functions are enriched among HOGs that were gained, duplicated, retained, or lost during evolution?

To answer this question, EAGLE combines:

- the species tree
- the HOG evolutionary histories
- propagated GO annotations from HogProp

This allows us to connect changes in gene families with changes in biological function.

## Running GO enrichment on a single branch

We will begin by analyzing the branch leading from **Archosauria** to **Aves**.

Archosauria includes crocodilians and birds, while Aves represents the ancestor of all modern birds.

By focusing on this branch, we can ask:

- Which functions became enriched during the origin of birds?
- Which functions were associated with gene family gains?
- Which functions were associated with gene family duplications?
- Which functions were preferentially lost?

For demonstration purposes, we will run EAGLE on this single branch. Later, we will inspect results generated across the entire phylogeny.

Path definitions for the files we will need to run EAGLE:

In [1]:
import os
import subprocess

common_data_path = '/home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData'

nwk_fn = os.path.join(common_data_path, 'Module2', 'species_tree.nwk')  # might need to be the species_tree_checked.nwk from FastOMA output
oxml_fn = os.path.join(common_data_path, 'Module2', 'fastoma_output', 'FastOMA_HOGs.orthoxml')

obo_fn = os.path.join(common_data_path, 'Module4', 'go-basic.obo')
hogprop_results_fn = os.path.join(common_data_path, 'Module4', 'precomputed_results', 'hogprop_output.h5')
#'../step2_propagation/hogprop_output.h5'

We need to save the species tree newick file from FastOMA results (species_tree_checked.nwk) as this has the same internal node names as the OrthoXML. Below we create a variable called nwk where we copy and paste the tree string. We then save the tree in a file in our directory.

In [2]:
#save the species tree in a string
nwk = '((Crocodylus_porosus:1,(((((((Aptenodytes_forsteri :1, Oreotrochilus_melanogaster :1)Neoaves_4c:1, Grus_americana :1)Neoaves_3c:1, Taeniopygia_guttata :1)Neoaves_2a:1, Columba_livia :1)Neoaves_1a:1, Phoenicopterus_ruber :1)Neoaves:1, Gallus_gallus :1)Neognathae:1, Struthio_camelus:1)Aves:0)Archosauria:1, Homo_sapiens:1)internal_0;'

#write species tree newick to current path
with open('species_tree.nwk', 'wt') as fp:
    print(nwk, file=fp)

We run EAGLE as follows:

In [23]:
os.makedirs("eagle_results", exist_ok=True)

OBO = obo_fn
ORTHOXML = oxml_fn
TREE = "species_tree.nwk"
HOGPROP = hogprop_results_fn
print(HOGPROP)
OUTDIR = "eagle_results"

cmd = [
    "eagle",
    "--obo", OBO,
    "--oxml", ORTHOXML,
    "--nwk", TREE,
    "--hogprop_results", HOGPROP,
    "--results", OUTDIR,
    "--include_genelist",
    "--skip_terminal",
    "--write_extant_genelist",
    "--parent_name", "Archosauria",
    "--child_name", "Aves"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

/home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData/Module4/precomputed_results/hogprop_output.h5
Loaded data in 13.417207 seconds
DONE! Took 535.298384s.



Now, to read the results we can analyse individual files / branches with the following:

In [24]:
from eagle.results import ResultsReader
!ls eagle_results

Archosauria_Aves_duplicated.extant_genelist.tsv.gz
Archosauria_Aves_duplicated.tsv.gz
Archosauria_Aves_gained.extant_genelist.tsv.gz
Archosauria_Aves_gained.tsv.gz
Archosauria_Aves_lost.extant_genelist.tsv.gz
Archosauria_Aves_lost.tsv.gz
Archosauria_Aves_retained.extant_genelist.tsv.gz
Archosauria_Aves_retained.tsv.gz


Let's look in the results to see which GO terms were enriched in the genes duplicated on the Aves branch:

In [25]:
r = ResultsReader('eagle_results/Archosauria_Aves_duplicated.tsv.gz')
r.header

{'eagle_version': '0.0.2',
 'time_run': '2026-06-21 11:49:47.363319',
 'time_taken': '124.382213',
 'branch_type': 'None',
 'event_type': 'duplicated',
 'parent_name': 'Archosauria',
 'child_name': 'Aves'}

In [27]:
# save the results in a dataframe and examine the first rows
df = r.read()
df.head()

,GO_ID,GO_name,GO_aspect,GO_depth,p_uncorrected,p_bonferroni,p_fdr_bh,study_count,study_size,study_ratio,study_proportion,population_count,population_size,population_ratio,population_proportion,fold_change,study_entries_with_go_term
0,GO:0010482,regulation of epidermal cell division,BP,5,1.801313e-10,4.179046e-08,6.138752e-09,232,232,232 / 232,1.0,15065,16583,15065 / 16583,0.908460,1.100763,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...
1,GO:0010839,negative regulation of keratinocyte proliferation,BP,7,1.916660e-10,4.446652e-08,6.138752e-09,232,232,232 / 232,1.0,15069,16583,15069 / 16583,0.908702,1.100471,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...
2,GO:0061436,establishment of skin barrier,BP,6,1.916660e-10,4.446652e-08,6.138752e-09,232,232,232 / 232,1.0,15069,16583,15069 / 16583,0.908702,1.100471,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...
3,GO:0010837,regulation of keratinocyte proliferation,BP,6,1.946628e-10,4.516178e-08,6.138752e-09,232,232,232 / 232,1.0,15070,16583,15070 / 16583,0.908762,1.100398,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...
4,GO:0045606,positive regulation of epidermal cell differen...,BP,6,1.946628e-10,4.516178e-08,6.138752e-09,232,232,232 / 232,1.0,15070,16583,15070 / 16583,0.908762,1.100398,HOG:0000029_2@Archosauria;HOG:0000032_2@Archos...


## Read precomputed EAGLE results

We have precomputed the results of EAGLE run on every evolutionary event set of genes in the tree (internal branches only). We can therefore read many at once:

In [7]:
from eagle.results import parse_to_concat
import gzip

goea_path = os.path.join(common_data_path, 'Module4', 'precomputed_results', 'eagle_results')
fns = list(map(lambda x: os.path.join(goea_path, x),
               filter(lambda x: not x.endswith('.extant_genelist.tsv.gz'),
                      os.listdir(goea_path))))
df = parse_to_concat(*fns)

#saves all the results from the different branches into 1 dataframe
with gzip.open('eagle_results_with_ancestral_genes.tsv.gz', 'wt') as fp:
    df.to_csv(fp, sep='\t', index=False)

with gzip.open('eagle_results.tsv.gz', 'wt') as fp:
    header = [x for x in list(df) if x != 'study_entries_with_go_term']
    df[header].to_csv(fp, sep='\t', index=False)

100%|██████████| 32/32 [00:00<00:00, 32.56it/s]


In [8]:
#examine the columns in the resulting file:
df.columns

Index(['GO_ID', 'GO_name', 'GO_aspect', 'GO_depth', 'p_uncorrected',
       'p_bonferroni', 'p_fdr_bh', 'study_count', 'study_size', 'study_ratio',
       'study_proportion', 'population_count', 'population_size',
       'population_ratio', 'population_proportion', 'fold_change',
       'study_entries_with_go_term', 'branch_type', 'event_type',
       'parent_name', 'child_name'],
      dtype='object')

## Filter results after Multiple Testing p-value correction

We now how a dataframe with all the EAGLE results, that is -- enriched GO terms for duplicated, gained, lost, and retained genes across the entire species tree. 

However, as we performed thousands of GO enrichments, we must take into consideration the multiple testing correction. Filter the table to only use signficant results with the FDR correction p <=0.05.

In [15]:
sig_df = df[df['p_fdr_bh']<=0.05]

sig_df.head()

,GO_ID,GO_name,GO_aspect,GO_depth,p_uncorrected,p_bonferroni,p_fdr_bh,study_count,study_size,study_ratio,...,population_count,population_size,population_ratio,population_proportion,fold_change,study_entries_with_go_term,branch_type,event_type,parent_name,child_name
0,GO:0051606,detection of stimulus,BP,2,6.811895e-305,4.672279e-301,4.672279e-301,488,1272,488 / 1272,...,849,13525,849 / 13525,0.062773,6.111704,HOG:0000780.1n_6&22748233402688@Neoaves_2a;HOG...,internal,lost,Neoaves_2a,Neoaves_3c
1,GO:0050911,detection of chemical stimulus involved in sen...,BP,5,2.678661e-294,1.837294e-290,9.186468e-291,309,1272,309 / 1272,...,337,13525,337 / 13525,0.024917,9.749419,HOG:0007767.1j.1r_7&22748155502656@Neoaves_2a;...,internal,lost,Neoaves_2a,Neoaves_3c
2,GO:0005549,odorant binding,MF,2,1.196698e-292,8.208154e-289,2.736051e-289,309,1189,309 / 1189,...,331,11792,331 / 11792,0.028070,9.258403,HOG:0007767.1j.1r_7&22748155502656@Neoaves_2a;...,internal,lost,Neoaves_2a,Neoaves_3c
3,GO:0004984,olfactory receptor activity,MF,4,3.832097e-286,2.628436e-282,6.571089e-283,309,1189,309 / 1189,...,337,11792,337 / 11792,0.028579,9.093565,HOG:0007767.1j.1r_7&22748155502656@Neoaves_2a;...,internal,lost,Neoaves_2a,Neoaves_3c
4,GO:0050907,detection of chemical stimulus involved in sen...,BP,4,3.791146e-268,2.600347e-264,5.200694e-265,315,1272,315 / 1272,...,381,13525,381 / 13525,0.028170,8.790949,HOG:0002221.1a_6&22748215859760@Neoaves_2a;HOG...,internal,lost,Neoaves_2a,Neoaves_3c


How many significantly enriched GO terms are associated with gained, duplicated, retained, and lost HOGs?

In [16]:
sig_df.groupby('event_type').size()

event_type
duplicated     6110
gained           87
lost          31063
retained        264
dtype: int64

# Using GO-Figure to reduce redundancy amongst GO enrichment results

## Why do we need GO-Figure?

GO enrichment analyses often return dozens or hundreds of significant GO terms.

Many of these terms are highly redundant because Gene Ontology is hierarchical. For example:

- feather development
- epithelial appendage development
- anatomical structure development

may all describe related biological processes.

GO-Figure reduces this redundancy by grouping semantically similar GO terms and visualizing the major biological themes.

This makes the enrichment results much easier to interpret.

## Reduce redundancy with GO-Figure

Use GO-Figure to cluster related GO terms and generate summary plots.

For each event type:

- gained
- duplicated
- retained
- lost

We will generate:

- A GO-Figure input file
- A GO-Figure summary plot

In [17]:
#create directories to store go-figure files

os.makedirs("gofigure_inputs", exist_ok=True)
os.makedirs("gofigure_outputs", exist_ok=True)

We will need to create a file for each event (lost, retained, duplicated, gained) and branch combination.

In [18]:
event_types = ['gained', 'lost', 'retained', 'duplicated']
branches = sig_df[['parent_name','child_name']].drop_duplicates().reset_index()
branches

,index,parent_name,child_name
0,0,Neoaves_2a,Neoaves_3c
1,139,Neognathae,Neoaves
2,195,Neoaves_1a,Neoaves_2a
3,377,Aves,Neognathae
4,3430,Neoaves,Neoaves_1a
5,4651,Archosauria,Aves
6,7547,internal_0,Archosauria
7,11368,Neoaves_3c,Neoaves_4c


In [19]:
#Create input tsv files needed for go-figure

for idx, row in branches.iterrows():

    parent_name = row['parent_name']
    child_name = row['child_name']

    for event_type in event_types:
        tmp_df = sig_df[
            (sig_df["event_type"] == event_type) &
            (sig_df['parent_name']==parent_name) &
            (sig_df['child_name']==child_name)
        ]
        
        gofigure_input = tmp_df[["GO_ID", "p_bonferroni"]]
        
        gofigure_input.to_csv(
            "./gofigure_inputs/{}_{}_{}.tsv".format(event_type, parent_name, child_name),
            sep="\t",
            index=False,
            header=False
        )

In [ ]:
#Run GO-Figure and create output plots

for idx, row in branches.iterrows():
    parent_name = row['parent_name']
    child_name = row['child_name']

    for event_type in event_types:

        cmd = [
            "python",
            "../gits/GO-Figure/gofigure.py",
            "-i", "gofigure_inputs/{}_{}_{}.tsv".format(event_type, parent_name, child_name),
            "-o", "gofigure_outputs/gofigure_{}_{}_{}".format(event_type, parent_name, child_name)
        ]
        print(cmd)
        subprocess.run(cmd, check=True)

# Examine major biological themes


Inspect the GO-Figure! plots and identify the most prominent functional categories.

Questions

Examine the GO enrichments of the evolutionary event categories on the Archosauria -> Aves branch.

- Which biological processes or cellular components appear most strongly enriched among gained HOGs?
- Which themes are associated with duplicated HOGs?
- Which functions are predominantly lost?
- Are there any biological themes that seem particularly relevant to bird evolution?